#### Импорты

In [ ]:
import pandas as pd
from math import ceil

from utils import *

In [ ]:
# Запускаем браузер (он откроется и останется висеть!)
print("🚀 Запускаем драйвер...")
driver = init_driver(stealth_mode=True, headless_mode=False)

# Теперь мы можем использовать крутые методы SeleniumBase прямо здесь!
TARGET_URL = 'https://2gis.ru/n_novgorod'
print(f"🌐 Переходим по адресу: {TARGET_URL}")

# uc_open_with_reconnect - спец. метод UC Mode для обхода жестких капч (переподключается, если запалили)
driver.uc_open_with_reconnect(TARGET_URL, reconnect_time=3)

print(f"✅ Страница загружена! Заголовок: {driver.title}")

#### Основной цикл сбора

In [ ]:
from math import ceil

# 0. Инициируем датафрейм для сохранения данных
df = pd.DataFrame()
save_path = r'data\ВУЗы 2ГИС.xlsx'

# 1. Узнаем, сколько всего мест в вышло в поисковой выдаче
total_places_str = get_text_by_css(driver, "._1xhlznaa", default="0")
total_places = int(total_places_str)

# По дефолту на 1 странице по 12 мест
total_pages = ceil(total_places / 12)

# 2. Инициация основного цикла
for _ in pages_tqdm(total_pages):

    # Собираем все места на странице
    places_on_page = get_visible_elements(driver)

    # Сбор по элементам этих мест
    for place_element in nested_tqdm(places_on_page):
        place_element.click()

        # Сбор информации по месту
        url, coords = get_place_coords(driver) # Кортеж координат (И ссылка до координат)
        attrs = get_main_attrs(place_element, coords, url)     # Буквально остальное

        # Сохранение
        if attrs:
            df = pd.concat([df, pd.DataFrame([attrs])], ignore_index=True)
            df.to_excel(save_path, index=False)
    
    # Переход на следующую страницу
    turn_page(driver)

print('✅ Поздравляю, парсинг окончен!')

#### Сбор со страницы вне цикла

In [ ]:
# Собираем все места на странице
places_on_page = get_visible_elements(driver)

# Сбор по элементам этих мест
for place_element in nested_tqdm(places_on_page):
    place_element.click()

    # Сбор информации по месту
    url, coords = get_place_coords(driver) # Кортеж координат (И ссылка до координат)
    attrs = get_main_attrs(place_element, coords, url)     # Буквально остальное

    # Сохранение
    if attrs:
        df = pd.concat([df, pd.DataFrame([attrs])], ignore_index=True)
        df.to_excel(save_path, index=False)